# Confidence-Driven Procurement Agent with ShopGraph

When a procurement agent auto-fills a purchase order, a wrong price is worse
than a missing price. This cookbook builds a LangChain agent that uses per-field
confidence scores to decide: act autonomously on high-confidence fields, route
low-confidence fields to human review.

**The core problem:** Extraction APIs return data. They don't tell you how much
to trust each field. A price pulled from a Schema.org tag is more reliable than
a price inferred from surrounding text. Without per-field confidence, the agent
either trusts everything (writes bad data into POs) or trusts nothing (you might
as well not have an agent).

**What ShopGraph adds:** Per-field confidence scores (0.0 to 1.0) on every
extracted field, plus server-side threshold filtering that scrubs uncertain
fields from the response before the agent can see them.

In [ ]:
%pip install requests langchain langchain-openai -q


In [ ]:
import os
import json
import requests

SHOPGRAPH_API_KEY = os.environ.get("SHOPGRAPH_API_KEY", "your-api-key")
SHOPGRAPH_API_URL = "https://shopgraph.dev/api/enrich"

# Confidence baselines by extraction method:
#   Schema.org / JSON-LD (tier 1): ~0.90-0.95
#   LLM extraction (tier 2):      ~0.65-0.80
#   Headless browser (tier 3):    ~0.50-0.70
#
# Scores reflect cross-tier agreement, not just one method's output
# probability. When two tiers agree on a price and one dissents,
# the confidence reflects that signal.


## Research mode and autofill mode

Procurement agents make two kinds of calls:

1. **Research mode.** A human reviews the output. The agent wants every field
   including low-confidence ones, so the human can judge what to trust.

2. **Autofill mode.** The agent writes data directly into a PO, inventory
   system, or RFQ response. No human reviews it. The agent wants only fields
   confident enough to act on. Anything below threshold should be absent,
   not flagged.

The difference matters because of context window contamination: if the agent
sees a low-confidence price in research mode, it may reference that price
later during autofill even though it shouldn't. Server-side filtering
(`strict_confidence_threshold`) removes the temptation entirely. The field
never enters the agent's context.

Both modes use the same tool: `enrich_product`. The agent picks the mode by
setting (or omitting) the `strict_confidence_threshold` argument. Omit it for
research; set it to 0.9 (or higher) for autofill.

In [ ]:
from langchain_core.tools import tool

@tool
def enrich_product(url: str, strict_confidence_threshold: float | None = None) -> str:
    """Extract product data from a URL. Returns per-field confidence scores and extraction provenance.

    Args:
        url: Product page URL to extract.
        strict_confidence_threshold: Optional 0.0-1.0 threshold. When set, fields with
            confidence below this value are removed from the response server-side before
            it reaches the agent. Use None for research mode (human reviews all fields).
            Use 0.9 for autofill mode (autonomous writes). Use 0.95 for contract pricing
            or regulated goods.

    Each field in the response includes a confidence score (0.0 to 1.0):
      0.90+     extracted from structured markup, high reliability
      0.70-0.89 extracted via LLM, moderate reliability
      below 0.70 inferred or partially matched, verify before relying on

    Returns:
        JSON string with product data and per-field confidence scores. In autofill mode,
        missing fields were below threshold and intentionally excluded — do not backfill
        them from other sources without flagging for human review.
    """
    body = {"url": url}
    if strict_confidence_threshold is not None:
        body["strict_confidence_threshold"] = strict_confidence_threshold
    response = requests.post(
        SHOPGRAPH_API_URL,
        headers={"Authorization": f"Bearer {SHOPGRAPH_API_KEY}"},
        json=body,
        timeout=30,
    )
    if not response.ok:
        return json.dumps({"error": True, "status": response.status_code, "url": url})
    return json.dumps(response.json(), indent=2)


## How confidence drives the autonomy decision

The agent doesn't decide trust levels. The API does.

When `enrich_product` is called without `strict_confidence_threshold`, it
returns all fields with confidence scores. The agent reports them to the
human. The human decides.

When `enrich_product` is called with `strict_confidence_threshold=0.9`, the
API scrubs every field below 0.9 before the response leaves the server. The
agent literally cannot see uncertain data. If it needs a field that was
scrubbed, it must stop and ask the human.

This is the difference between "the agent is trusted to ignore bad data"
and "the agent cannot see the bad data." For autonomous writes into
procurement systems, the second is the only defensible posture.

In [ ]:
# Research mode: extract all fields with confidence scores (no threshold set)
MOGLIX_URL = "https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8"

result = json.loads(enrich_product.invoke({"url": MOGLIX_URL}))

if "error" in result:
    print(f"Extraction failed: {result}")
else:
    product = result.get("product", result)
    shopgraph_meta = product.get("_shopgraph", {})
    field_confidence = shopgraph_meta.get("field_confidence", {})

    print("Extracted fields with confidence:\n")
    for field, conf in sorted(field_confidence.items(), key=lambda x: -x[1]):
        status = "OK" if conf >= 0.85 else "VERIFY" if conf >= 0.50 else "SKIP"
        print(f"  {field:20s}  confidence: {conf:.2f}  [{status}]")


## LangChain agent with threshold-based autonomy routing

The system prompt teaches the agent when to pass `strict_confidence_threshold`:

- **Researching a product?** Call `enrich_product` with no threshold. Show all
  fields with confidence. Flag anything below 0.85.
- **Auto-filling a PO or writing into a system?** Call `enrich_product` with
  `strict_confidence_threshold=0.9`. If a required field is missing from the
  response, stop and ask the human — a missing field means the API scrubbed it
  because confidence was below threshold.

The agent never mixes modes in a single operation. Research first, confirm the
fields, then autofill.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o", temperature=0)

tools = [enrich_product]

system_prompt = """You are a procurement assistant.

Use enrich_product for every extraction. Set strict_confidence_threshold based
on downstream use: omit it for research (human reviews all fields), set it to
0.9 for autofill mode (autonomous writes, fields below threshold removed
server-side). Never mix modes in a single operation.

RESEARCH MODE — call enrich_product(url) with no threshold:
Use when the human is researching a product, comparing suppliers, or asking
you to summarize what's available at a URL. Return all fields with their
confidence scores. Flag any field below 0.85 as "verify before relying on."
The human makes the trust decision.

AUTOFILL MODE — call enrich_product(url, strict_confidence_threshold=0.9):
Use when the human asks you to fill a purchase order, update an inventory
record, or write extracted values into any downstream system without human
review. Default threshold: 0.9. Use 0.95 for contract pricing or regulated
goods.

If a required field is missing from the autofill response, do NOT invent it,
guess it, or pull it from earlier in the conversation. Stop and tell the
human: "This field was below the confidence threshold and needs manual
verification."

If the task spans both research and autofill, do the research call first
(no threshold), confirm the fields with the human, then make the autofill
call (with threshold)."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


## Demo: research then autofill

Watch the tool calls. The agent should:

1. Call `enrich_product` with no threshold (research mode, all fields visible)
2. Report confidence scores to you
3. Call `enrich_product` with `strict_confidence_threshold=0.9` (autofill mode)
4. Report which fields survived the threshold and which were scrubbed

If a field the PO needs was scrubbed, the agent should stop and ask rather
than invent a value.

In [ ]:
result = executor.invoke({
    "input": (
        f"Research this product: {MOGLIX_URL}. "
        f"Show me all extracted fields with their confidence scores. "
        f"Then, assuming the research looks good, prepare a PO line item "
        f"for 3 units using only fields with confidence above 0.9."
    )
})

print("\n" + "=" * 60)
print("AGENT OUTPUT:")
print("=" * 60)
print(result["output"])


## Why server-side filtering matters

Client-side filtering (checking confidence after the response arrives)
still lets the agent see low-confidence data. In a long context window,
the agent may reference a scrubbed price field from earlier in the
conversation because it was visible during the research step.

Server-side filtering via `strict_confidence_threshold` removes the field
from the API response entirely. The agent cannot reference what it never
received.

For autonomous writes into procurement systems, inventory databases, or
any system where a wrong value has downstream consequences: use server-side
filtering. The pattern is `enrich_product(url, strict_confidence_threshold=0.9)`,
with the threshold matched to the stakes of the write.

## Reference

**Pricing:**
Playground: 50 extractions/month, no signup required.
Starter: $99/month for 10K calls with API key.

**Confidence baselines by extraction method:**
| Method | Typical range | When used |
|---|---|---|
| Schema.org / JSON-LD (tier 1) | 0.90 to 0.95 | Site has structured markup |
| LLM extraction (tier 2) | 0.65 to 0.80 | No structured markup, page has readable product content |
| Headless browser (tier 3) | 0.50 to 0.70 | Content requires JavaScript rendering |

Scores reflect cross-tier agreement. When multiple tiers extract the same
field, agreement raises confidence; disagreement lowers it.

**Additional API options (not used in this cookbook):**
- AgentReady scoring: `?include_score=true` returns a 0-100 readiness score
- UCP output: `?format=ucp` for Universal Commerce Protocol schema
- Leaderboard: [shopgraph.dev/leaderboard](https://shopgraph.dev/leaderboard) shows which sites extract successfully

**Related:**
- [Vercel AI SDK example](https://github.com/vercel/ai) for client-side
  confidence rendering in a Next.js UI (complementary pattern: this cookbook
  does server-side filtering, that example does client-side rendering)